In [1]:
from google.colab import drive
import os

# Check if Google Drive is already mounted
if not os.path.exists('/content/drive'):
  drive.mount('/content/drive')
else:
  print("Google Drive already mounted.")

dataFolder = "/content/drive/MyDrive/Colab_Notebooks/00_lichess/03_Triangle/Join_v1_v2"

!ls {dataFolder} -lh


Mounted at /content/drive
total 2.7M
-rw------- 1 root root 1.6M Nov 14 15:30 'Copy of 20251113_lichess_db_puzzle_mate_triangle.csv'
-rw------- 1 root root 5.8K Nov 14 15:31 'Copy of RandomSelection4Review.ipynb'
-rw------- 1 root root 1.2M Nov 14 15:28  lichess_db_puzzle_mate_triangle_1st_review.csv


In [7]:
import pandas as pd

file1_path = f"{dataFolder}/Copy of 20251113_lichess_db_puzzle_mate_triangle.csv"
file2_path = f"{dataFolder}/lichess_db_puzzle_mate_triangle_1st_review.csv"

df1 = pd.read_csv(file1_path)
df2 = pd.read_csv(file2_path, delimiter='|')

In [6]:
print(df2.head(5))

  PuzzleId|FEN|Moves|Rating|RatingDeviation|Popularity|NbPlays|Themes|GameUrl|OpeningTags|Valid|puzzle_ref|Reason for Invalidation|Reviewed
0  C3H2A|r4rkR/pp2ppB1/5np1/2p5/3p4/3P1qP1/PPPQ1P...                                                                                       
1  nO7kp|5k1R/pp2bp2/2p1p1p1/3qP3/8/2P1Q3/PP3PP1/...                                                                                       
2  8UkPQ|rq4kr/1b1p1R1B/p2Np3/1pBnP1p1/7Q/8/PP4P1...                                                                                       
3  cB0hO|8/pp3Rkr/2p5/7Q/1PP5/6PK/1q2p2P/8 b - - ...                                                                                       
4  eL6JE|R1k3rr/1Rpp1p1p/1p6/3Pq3/4P3/2PQB2p/5PP1...                                                                                       


In [8]:
import pandas as pd

file1_path = f"{dataFolder}/Copy of 20251113_lichess_db_puzzle_mate_triangle.csv"
file2_path = f"{dataFolder}/lichess_db_puzzle_mate_triangle_1st_review.csv"

df1 = pd.read_csv(file1_path, delimiter=',')
df2 = pd.read_csv(file2_path, delimiter='|')

# Identify common columns (excluding 'PuzzleId')
common_cols = [col for col in df1.columns if col in df2.columns and col != 'PuzzleId']

# Perform an outer join, adding suffixes to common columns
merged_df = pd.merge(df1, df2, on='PuzzleId', how='outer', suffixes=('_df1', '_df2'))

# For each common column, prioritize values from df1
for col in common_cols:
    merged_df[col] = merged_df[col + '_df1'].fillna(merged_df[col + '_df2'])
    # Drop the suffixed columns
    merged_df = merged_df.drop(columns=[col + '_df1', col + '_df2'])

# Get unique columns from df1 and df2 that are not 'PuzzleId' and not in common_cols
df1_unique_to_df1 = [col for col in df1.columns if col not in common_cols and col != 'PuzzleId']
df2_unique_to_df2 = [col for col in df2.columns if col not in common_cols and col != 'PuzzleId']

# Construct the final desired column order
# Start with 'PuzzleId'
final_column_order = ['PuzzleId']

# Add the combined common columns
final_column_order.extend(common_cols)

# Add columns unique to df1
final_column_order.extend(df1_unique_to_df1)

# Add columns unique to df2
final_column_order.extend(df2_unique_to_df2)

# Ensure no duplicate columns in the final list and only include columns actually present in merged_df
final_column_order = [col for col in pd.unique(final_column_order) if col in merged_df.columns]

# Apply the new column order
merged_df = merged_df[final_column_order]


print(f"Shape of df1: {df1.shape}")
print(f"Shape of df2: {df2.shape}")
print(f"Shape of merged_df: {merged_df.shape}")
print("\nFirst 5 rows of the merged DataFrame:")
print(merged_df.head())
print("\nNumber of nulls in 'PuzzleId' column after merge (should be 0):")
print(merged_df['PuzzleId'].isnull().sum())
print("\nColumn names and their order in the merged DataFrame:")
print(merged_df.columns.tolist())

Shape of df1: (7756, 12)
Shape of df2: (5750, 14)
Shape of merged_df: (7936, 15)

First 5 rows of the merged DataFrame:
  PuzzleId                                                FEN  \
0    00qX2  1r2r2k/Q1R4p/4q1p1/4Bb2/8/8/PP3PPP/3R2K1 b - -...   
1    012uA  r4r2/p1p2k1R/6R1/3n1pB1/1q1Pb3/8/1P2Q1PP/6K1 b...   
2    01YV3  1r3rk1/p3R2p/3p2pQ/3q4/3b4/3p4/PP3PPP/1RB4K b ...   
3    03vEG    8/6p1/1q2p2p/3kP3/1nR4Q/7P/r4PP1/6K1 b - - 0 39   
4    04UtS  7k/1pq3p1/4R2p/pBP5/1P4R1/P1Q2r1P/5rP1/6K1 w -...   

                           Moves  Rating  RatingDeviation  Popularity  \
0            e6e5 c7h7 h8g8 a7f7  1195.0             79.0        92.0   
1                      f7g6 e2h5  1139.0             77.0        84.0   
2  f8f2 h6h7 g8f8 c1h6 d4g7 h7g7  1551.0             74.0        97.0   
3                      b4c6 h4e4  1157.0             75.0        99.0   
4                      g2f3 c7h2   838.0             84.0       100.0   

   NbPlays                           Themes  \
0  

/tmp/ipython-input-3836948702.py:39: FutureWarning: unique with argument that is not not a Series, Index, ExtensionArray, or np.ndarray is deprecated and will raise in a future version.
  final_column_order = [col for col in pd.unique(final_column_order) if col in merged_df.columns]


In [10]:
merged_df.sort_values("Reviewed").to_csv(f"{dataFolder}/lichess_db_puzzle_mate_triangle_merged.csv", index=False)